# EdgeForge AI on a free Colab GPU

One coarse layout in, a labelled computer-vision dataset out. This notebook runs the real engine (the FastAPI app in `src/app/main.py`) on Colab's free T4 and calls it the way the web UI does.

**Before you run it:** *Runtime → Change runtime type* → **T4 GPU**, and **Runtime version 2026.07** (Python 3.12). The pinned `numpy<2` has no wheels for Python 3.13, which newer runtimes use.

The first run downloads about 10 GB of model weights (SDXL base, the SDXL Canny ControlNet, the SDXL VAE and YOLOv8n), which takes a few minutes. Each 1024×1024 image then takes about 80 seconds on a T4.

In [ ]:
import sys
assert sys.version_info < (3, 13), "Pick Runtime > Change runtime type > Runtime version 2026.07 (Python 3.12)."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!git clone -q https://github.com/yashsinha1702/edgeforge-ai.git /content/edgeforge-ai
!pip -q install -r /content/edgeforge-ai/requirements.txt "transformers==4.44.2" "accelerate==0.34.2"
# pip ends with a red list of conflicts: Colab packages that EdgeForge never imports (jax, gradio, ...)
# want numpy 2 or a newer huggingface_hub. That is expected; the next cells check that the engine starts.
# diffusers 0.25 imports peft when it is installed, and Colab's peft needs a newer huggingface_hub
!pip -q uninstall -y peft

## Start the engine

The same server the Dockerfile starts, run in the background. On the first run it downloads the weights (a few minutes), then loading them takes about a minute. The cell prints the engine's progress every 30 seconds and returns once the engine answers on port 8000. If the engine dies (out of memory, for example), the cell prints its log and stops.

In [ ]:
import os, subprocess, sys, time, urllib.request

LOG = "/content/engine.log"

def engine_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/docs", timeout=5)
        return True
    except OSError:
        return False

def last_log_line():
    text = open(LOG, errors="replace").read().replace("\r", "\n").strip()
    return text.splitlines()[-1][:120] if text else ""

if engine_up():
    print("The engine is already running.")
else:
    # Its own session, so interrupting a later cell leaves it running; unbuffered, so the log is live
    engine = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "src.app.main:app", "--host", "127.0.0.1", "--port", "8000"],
        cwd="/content/edgeforge-ai", env={**os.environ, "PYTHONPATH": "src", "PYTHONUNBUFFERED": "1"},
        stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    start = time.time()
    next_report = start + 30
    while not engine_up():
        if engine.poll() is not None:
            print(open(LOG, errors="replace").read()[-3000:])
            raise RuntimeError(f"The engine stopped with exit code {engine.returncode}; its log is above.")
        if time.time() >= next_report:
            print(f"{time.time() - start:4.0f}s  {last_log_line()}")
            next_report += 30
        time.sleep(5)
    print(f"Engine ready after {time.time() - start:.0f}s")

## Generate a small dataset

`docs/layout.png` is a coarse white car on black, the kind of input the README describes. The batch endpoint remixes the layout for every image (1–3 cars, shifted, scaled and flipped), samples a condition and a seed, generates, and labels the result with YOLOv8.

In [ ]:
# Four images take about five minutes on a T4
!curl -sS --fail -X POST http://127.0.0.1:8000/generate_batch \
  -F "intent=a car on a highway" -F "batch_size=4" \
  -F "control_image=@/content/edgeforge-ai/docs/layout.png" \
  -o /content/edgeforge_dataset.zip && unzip -o -q /content/edgeforge_dataset.zip -d /content/dataset && ls -R /content/dataset | head -20

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

COCO = {0: "person", 2: "car", 3: "motorcycle", 4: "airplane", 5: "bus", 7: "truck"}  # other IDs print as numbers
root = Path("/content/dataset")
prompts = {p["image"]: p for p in json.loads((root / "prompts.json").read_text())}
tiles = []
for img_path in sorted((root / "images").glob("*.png")):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    found = []
    for line in (root / "labels" / (img_path.stem + ".txt")).read_text().split("\n"):
        if not line.strip():
            continue
        cls, x, y, w, h = line.split()
        x, w = float(x) * im.width, float(w) * im.width
        y, h = float(y) * im.height, float(h) * im.height
        draw.rectangle([x - w / 2, y - h / 2, x + w / 2, y + h / 2], outline="#ffd400", width=5)
        found.append(COCO.get(int(cls), cls))
    print(f'{img_path.name} | {prompts[f"images/{img_path.name}"]["condition"]} | {", ".join(found) or "no boxes"}')
    tiles.append(im.resize((512, 512)))
grid = Image.new("RGB", (512 * len(tiles), 512))
for i, t in enumerate(tiles):
    grid.paste(t, (512 * i, 0))
display(grid)

The zip is a YOLO-format dataset: `images/`, `labels/` (one `class x_center y_center width height` line per box, normalised) and `prompts.json`, which records the prompt, condition and seed behind each image. The class IDs are COCO indices from the pretrained YOLOv8n, so spot-check the labels before you train on them.